In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings("ignore")
# Loading the process data from feature ready file from Phase - 3. 

df = pd.read_csv('../data/processed/application_train_processed.csv')

print(f"Processed data loaded : {df.shape}")
print(f"Expected (307511 , 105)")

Processed data loaded : (307511, 105)
Expected (307511 , 105)


In [2]:
# ── LOAD FEATURE LIST AND SEPARATE X, y ───────────────────────────────────────
# Read the exact 102 features confirmed in Step 3.5

with open('../docs/feature_list.txt', 'r') as f:
    lines = f.readlines()

# skip header lines and extract only feature list 
final_features = [line.strip() for line in lines
                  if line.strip() and not line.startswith('Final')
                  and not line.startswith('Generated')]

print(f"Features loaded from docs/feature_list.txt : {len(final_features)}")
print(f"Expected (102)")

# Seperate features (X) and target (y)
X = df[final_features]
y = df['TARGET']    

ids = df['SK_ID_CURR']
print(f"X shape : {X.shape}")
print(f"Y shape : {y.shape}")
print(f"Overall default rate :{y.mean()*100:.2f}%")

Features loaded from docs/feature_list.txt : 102
Expected (102)
X shape : (307511, 102)
Y shape : (307511,)
Overall default rate :8.07%


In [3]:
print(lines)

['Final Feature List ï¿½ 102 features\n', 'Generated: Step 3.5 Feature Selection\n', '\n', 'NAME_CONTRACT_TYPE\n', 'FLAG_OWN_CAR\n', 'FLAG_OWN_REALTY\n', 'CNT_CHILDREN\n', 'AMT_INCOME_TOTAL\n', 'AMT_CREDIT\n', 'AMT_ANNUITY\n', 'AMT_GOODS_PRICE\n', 'NAME_EDUCATION_TYPE\n', 'REGION_POPULATION_RELATIVE\n', 'DAYS_BIRTH\n', 'DAYS_EMPLOYED\n', 'DAYS_REGISTRATION\n', 'DAYS_ID_PUBLISH\n', 'OWN_CAR_AGE\n', 'FLAG_WORK_PHONE\n', 'OCCUPATION_TYPE\n', 'CNT_FAM_MEMBERS\n', 'REGION_RATING_CLIENT\n', 'REGION_RATING_CLIENT_W_CITY\n', 'HOUR_APPR_PROCESS_START\n', 'REG_REGION_NOT_WORK_REGION\n', 'LIVE_REGION_NOT_WORK_REGION\n', 'REG_CITY_NOT_LIVE_CITY\n', 'REG_CITY_NOT_WORK_CITY\n', 'LIVE_CITY_NOT_WORK_CITY\n', 'ORGANIZATION_TYPE\n', 'EXT_SOURCE_1\n', 'EXT_SOURCE_2\n', 'EXT_SOURCE_3\n', 'APARTMENTS_MEDI\n', 'BASEMENTAREA_MEDI\n', 'YEARS_BEGINEXPLUATATION_MEDI\n', 'YEARS_BUILD_MEDI\n', 'COMMONAREA_MEDI\n', 'ELEVATORS_MEDI\n', 'ENTRANCES_MEDI\n', 'FLOORSMAX_MEDI\n', 'FLOORSMIN_MEDI\n', 'LANDAREA_MEDI\n', '

In [4]:
# ── STRATIFIED TRAIN / VALIDATION / TEST SPLIT ────────────────────────────────
# 60% train, 20% validation, 20% test
# Stratify on TARGET to preserve 8.07% default rate in every split

# Step 1 — Split off the test set first (20%)
X_temp, X_test, y_temp, y_test, ids_temp, ids_test = train_test_split(
    X, y, ids,
    test_size=0.20,
    stratify=y,
    random_state=42
)

# Step 2 — Split the remaining 80% into train (60% of total) and validation (20% of total)
# 0.25 of the remaining 80% = 20% of the original total
X_train, X_val, y_train, y_val, ids_train, ids_val = train_test_split(
    X_temp, y_temp, ids_temp,
    test_size=0.25,
    stratify=y_temp,
    random_state=42
)

print("=== SPLIT COMPLETE ===\n")
print(f"Train:      {X_train.shape[0]:>7,} rows  ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"Validation: {X_val.shape[0]:>7,} rows  ({X_val.shape[0]/len(X)*100:.1f}%)")
print(f"Test:       {X_test.shape[0]:>7,} rows  ({X_test.shape[0]/len(X)*100:.1f}%)")
print(f"Total:      {X_train.shape[0]+X_val.shape[0]+X_test.shape[0]:>7,} rows")

print(f"\n=== DEFAULT RATE VERIFICATION (should all be ~8.07%) ===")
print(f"Overall:    {y.mean()*100:.2f}%")
print(f"Train:      {y_train.mean()*100:.2f}%")
print(f"Validation: {y_val.mean()*100:.2f}%")
print(f"Test:       {y_test.mean()*100:.2f}%")

=== SPLIT COMPLETE ===

Train:      184,506 rows  (60.0%)
Validation:  61,502 rows  (20.0%)
Test:        61,503 rows  (20.0%)
Total:      307,511 rows

=== DEFAULT RATE VERIFICATION (should all be ~8.07%) ===
Overall:    8.07%
Train:      8.07%
Validation: 8.07%
Test:       8.07%


In [5]:
# ── SAVE SPLITS TO DISK ────────────────────────────────────────────────────────
# Each split saved separately — the standard handoff pattern for Phase 4

import os
os.makedirs('../data/processed/splits', exist_ok=True)

# Combine X, y, and ids for each split before saving
train_df = X_train.copy()
train_df['TARGET'] = y_train
train_df['SK_ID_CURR'] = ids_train

val_df = X_val.copy()
val_df['TARGET'] = y_val
val_df['SK_ID_CURR'] = ids_val

test_df = X_test.copy()
test_df['TARGET'] = y_test
test_df['SK_ID_CURR'] = ids_test

train_df.to_csv('../data/processed/splits/train.csv', index=False)
val_df.to_csv('../data/processed/splits/validation.csv', index=False)
test_df.to_csv('../data/processed/splits/test.csv', index=False)

print("=== SPLITS SAVED ===\n")
print(f"train.csv:      {train_df.shape}")
print(f"validation.csv: {val_df.shape}")
print(f"test.csv:       {test_df.shape}")

print(f"\nSaved to: data/processed/splits/")
print(f"All three files gitignored (data/processed/ already excluded)")

=== SPLITS SAVED ===

train.csv:      (184506, 104)
validation.csv: (61502, 104)
test.csv:       (61503, 104)

Saved to: data/processed/splits/
All three files gitignored (data/processed/ already excluded)
